In [ ]:
!pip install -q python-dotenv

import os
from google.colab import drive
from dotenv import load_dotenv

drive.mount('/content/drive')

# Указываем точный путь к .env на Google Диске
ENV_PATH = "/content/drive/MyDrive/67.env"
load_dotenv(ENV_PATH)

# Пример чтения переменной:
# hf_token = os.getenv("HF_TOKEN")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


True

In [ ]:
import os
import re
import json
import random
import pandas as pd
from openai import OpenAI
from dotenv import load_dotenv
from datasets import load_dataset

# 1. Загружаем именно ваш файл
success = load_dotenv()
print("Удалось загрузить файл?", success)

# 2. Проверяем, видит ли система переменную API_KEY
api_key = os.getenv("API_KEY")
print("API_KEY успешно прочитан?", "Да!" if api_key else "Нет (значение пустое)")

if api_key:
    # Безопасный вывод первых 4 символов ключа для проверки работоспособности
    print(f"Начало вашего ключа: {api_key[:4]}...")

Удалось загрузить файл? False
API_KEY успешно прочитан? Да!
Начало вашего ключа: sk-8...


In [ ]:

API_KEY = os.getenv("API_KEY")
#тут файл надо создать ".env" и в него прописать API_KEY=your_api_key и др
BASE_URL = os.getenv("API_BASE_URL", "http://deepcode.ci.nsu.ru/api")
# MODEL_NAME = os.getenv("LLM_MODEL")
MODEL_NAME = "deepseek-ai/DeepSeek-V4-Flash"
client = OpenAI(
    api_key=API_KEY,
    base_url=BASE_URL
)


In [ ]:
def clean_deepseek_output(text):
    if not text:
        return ""
    cleaned = re.sub(r'<think>.*?</think>', '', text, flags=re.DOTALL)
    cleaned = cleaned.replace('"', '').replace('"', '').strip()
    return cleaned

def call_deepseek(system_prompt, user_text, temperature=1):
    """Отправка запроса к серверу НГУ."""
    try:
        response = client.chat.completions.create(
            model=MODEL_NAME,
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_text}
            ],
            temperature=temperature,
            top_p=0.8,
            extra_body={
                "repetition_penalty": 1.0,
                "presence_penalty": 1.5,
            }
        )
        raw_text = response.choices[0].message.content
        return clean_deepseek_output(raw_text)
    except Exception as e:
        print(f" [API Error]: {e}")
        return None

In [ ]:
PROMPTS_SUBJECT = [
    "Rewrite the email subject line to increase Open Rate. Make it intriguing. CRITICAL: You MUST preserve all original tags (e.g., [ORGANIZATION], [NAME]) EXACTLY as they appear, keeping exact case and brackets. Return ONLY the new subject line in English, no quotes. Do not include labels like 'Subject:', 'Body:', or any metadata in your response. Output ONLY the mutated text.",

    "Paraphrase the email subject line in a formal corporate tone. CRITICAL: You MUST preserve all original tags (e.g., [ORGANIZATION], [NAME]) EXACTLY as they appear, keeping exact case and brackets. Return ONLY the new subject line in English, no quotes. Do not include labels like 'Subject:', 'Body:', or any metadata in your response. Output ONLY the mutated text.",

    "Make the email subject line sound urgent (Urgency trigger). CRITICAL: You MUST preserve all original tags (e.g., [ORGANIZATION], [NAME]) EXACTLY as they appear, keeping exact case and brackets. Return ONLY the new subject line in English, no quotes. Do not include labels like 'Subject:', 'Body:', or any metadata in your response. Output ONLY the mutated text."
]

PROMPTS_BODY = [
    # 1. Формальный корпоративный (классика)
    "Rewrite the following email paragraph to sound highly professional and corporate. "
    "CRITICAL: You MUST preserve every single placeholder tag (e.g., [ORGANIZATION], [NAME], [FILE_NAME], <|PHONE|>, <|URL|>) EXACTLY as it appears in the original text. Do not change their case, remove their brackets, or omit them. "
    "Do not include labels like 'Subject:', 'Body:', or any metadata in your response. Output ONLY the mutated text.",

    # 2. Краткий и прямой (без воды)
    "Rewrite the following email paragraph to be extremely concise and direct. Cut any fluff and get straight to the point. "
    "CRITICAL: You MUST preserve every single placeholder tag (e.g., [ORGANIZATION], [NAME], [FILE_NAME], <|PHONE|>, <|URL|>) EXACTLY as it appears in the original text. Do not omit any tags, even when shortening the text. Do not change their case or remove their brackets. "
    "Do not include labels like 'Subject:', 'Body:', or any metadata in your response. Output ONLY the mutated text.",

    # 3. Дружелюбный и неформальный
    "Rewrite the following email paragraph in a casual, friendly, and approachable tone. "
    "CRITICAL: You MUST preserve every single placeholder tag (e.g., [ORGANIZATION], [NAME], [FILE_NAME], <|PHONE|>, <|URL|>) EXACTLY as it appears in the original text. Do not change their case, remove their brackets, or omit them. "
    "Do not include labels like 'Subject:', 'Body:', or any metadata in your response. Output ONLY the mutated text.",

    # 4. Структурированный / Экшен-ориентированный
    "Rewrite the following email paragraph to be highly actionable. Use clear, short sentences or restructure into a more readable format. "
    "CRITICAL: You MUST preserve every single placeholder tag (e.g., [ORGANIZATION], [NAME], [FILE_NAME], <|PHONE|>, <|URL|>) EXACTLY as it appears in the original text. Do not change their case, remove their brackets, or omit them. "
    "Do not include labels like 'Subject:', 'Body:', or any metadata in your response. Output ONLY the mutated text."
]

In [ ]:
2import os
import json

def load_processed_ids(output_file):
    processed = set()
    if not os.path.exists(output_file):
        return processed
    with open(output_file, "r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                try:
                    data = json.loads(line)
                    processed.add((data["email_id"], data["variant_index"]))
                except json.JSONDecodeError:
                    continue
    return processed

In [10]:
def process_meajor_dataset_safe(output_jsonl="mutated_spam_dataset.jsonl", num_variants=2, max_items_per_run=100):
    print("Загрузка датасета MeAJOR...")
    dataset = load_dataset("simlab-vs/meajor_cleaned_preprocessed")
    df = dataset["train"].to_pandas()

    processed_keys = load_processed_ids(output_jsonl)
    print(f"Уже готово в базе: {len(processed_keys)} записей.")

    newly_generated = 0

    with open(output_jsonl, "a", encoding="utf-8") as f:
        for index, row in df.iterrows():
            if newly_generated >= max_items_per_run:
                print(f"\n[Пауза] Достигнут лимит запуска ({max_items_per_run} шт.).")
                break

            email_id = int(row.get("id", index))
            orig_subject = str(row.get("subject", ""))
            orig_body = str(row.get("body", ""))

            # Защита от NaN и слишком коротких текстов
            if pd.isna(row.get("body")) or orig_body.strip().lower() == "nan" or len(orig_body.strip()) < 15:
                print(f"Пропуск Email ID {email_id}: мусор или пустой текст.")
                continue

            if not orig_subject or not orig_body:
                continue

            for variant_num in range(num_variants):
                if (email_id, variant_num) in processed_keys:
                    continue

                # Рандомизация температуры (как ты и сделал)
                current_temp = random.uniform(0.8, 1.0)

                # 1. Случайный выбор промптов
                chosen_subject_prompt = random.choice(PROMPTS_SUBJECT)
                chosen_body_prompt = random.choice(PROMPTS_BODY)

                # 2. Генерация темы
                mutated_subject = call_deepseek(
                    system_prompt=chosen_subject_prompt,
                    user_text=orig_subject,
                    temperature=current_temp
                )

                # 3. Генерация тела письма
                mutated_body = call_deepseek(
                    system_prompt=chosen_body_prompt,
                    user_text=orig_body,
                    temperature=current_temp
                )

                # Очистка на всякий случай (срезаем случайные пробелы и артефакты с краев)
                mutated_subject = mutated_subject.strip() if mutated_subject else orig_subject
                mutated_body = mutated_body.strip() if mutated_body else orig_body

                print(f"DEBUG Тема: {mutated_subject}")
                print(f"DEBUG Тело: {mutated_body}")

                entry = {
                    "email_id": email_id,
                    "variant_index": variant_num,
                    "original_subject": orig_subject,
                    "original_body": orig_body,
                    "ai_mutated_subject": mutated_subject,
                    "ai_mutated_body": mutated_body,
                }

                f.write(json.dumps(entry, ensure_ascii=False) + "\n")
                f.flush()

                processed_keys.add((email_id, variant_num))
                newly_generated += 1

                if newly_generated >= max_items_per_run:
                    break

    print(f"\nГотово! Файл {output_jsonl} успешно обновлен.")

if __name__ == "__main__":
    process_meajor_dataset_safe(max_items_per_run=10)

Загрузка датасета MeAJOR...
Уже готово в базе: 152 записей.
Пропуск Email ID 9: мусор или пустой текст.
DEBUG Тема: Unlock the Secret Before It’s Gone
DEBUG Тело: <|EMOJI|>FFFFA1<|EMOJI|>FFFFD8<|EMOJI|>FFFFC1<|EMOJI|>FFFFF7<|EMOJI|>FFFFC0<|EMOJI|>FFFFE5<|EMOJI|>FFFFC0<|EMOJI|>FFFFCE <|EMOJI|>FFFFB4<|EMOJI|>FFFFA9<|EMOJI|>FFFFB1<|EMOJI|>FFFFB8<|EMOJI|>FFFFB3<|EMOJI|>FFFFAA
<|EMOJI|>FFFFB4<|EMOJI|>FFFFEB<|EMOJI|>FFFFC3<|EMOJI|>FFFFE2 <|EMOJI|>FFFFB0<|EMOJI|>FFFFA1<|EMOJI|>FFFFB4<|EMOJI|>FFFFC9<|EMOJI|>FFFFA1<|EMOJI|>FFFFD8

We acknowledge receipt of your recent communication and thank you for bringing this matter to our attention. Please be assured that we are addressing the issue with the utmost priority. We will provide you with a detailed update within the next business day. Should you require further assistance, please do not hesitate to contact our support team.


KeyboardInterrupt: 